# Cost per Token Analysis

This notebook analyzes **cost per token** across different workload levels - a more standardized metric than cost per request.

**Key Advantages of Cost per Token:**
- Independent of request size variations
- Standardized comparison across different use cases
- More accurate for dynamic hardware selection
- Better reflects actual processing costs

**Workload Levels:** 10, 50, 100, 500, 1000, 5000+ tokens/hour
**Primary Metric:** Cost per token ($)

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('default')
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 11

print("Libraries imported successfully!")

In [ ]:
# Cost assumptions
cost_assumptions = {
    ('cpu', 1): {'hourly_cost': 0.05, 'idle_multiplier': 1.0},
    ('cpu', 2): {'hourly_cost': 0.10, 'idle_multiplier': 1.0},
    ('cpu', 4): {'hourly_cost': 0.20, 'idle_multiplier': 1.0},
    ('cpu', 8): {'hourly_cost': 0.40, 'idle_multiplier': 1.0},
    ('cuda', 25): {'hourly_cost': 0.50, 'idle_multiplier': 0.8},
    ('cuda', 50): {'hourly_cost': 1.00, 'idle_multiplier': 0.8},
    ('cuda', 75): {'hourly_cost': 1.50, 'idle_multiplier': 0.8},
    ('cuda', 100): {'hourly_cost': 2.00, 'idle_multiplier': 0.8},
}

def get_device_key(record):
    variant = record['variant']
    if variant == 'cpu':
        return ('cpu', record['cpu_cores'])
    elif variant == 'cuda':
        return ('cuda', record['gpu_percentage'])

# Load Q4_K_M model data
with open('parsed_logs/night_logs_6_with_model_info.json', 'r') as f:
    data = json.load(f)

q4_models = [record for record in data if record.get('model_quant', '') == 'Q4_K_M']
print(f"Found {len(q4_models)} Q4_K_M model records")

# Process data
processed_data = []
for record in q4_models:
    try:
        device_key = get_device_key(record)
        if device_key not in cost_assumptions:
            continue

        costs = cost_assumptions[device_key]
        throughput = record.get('throughput_mean', 0)  # tokens per second
        variant, config_value = device_key

        hw_config = f"CPU {config_value} cores" if variant == 'cpu' else f"GPU {config_value}%"

        processed_data.append({
            'hardware_config': hw_config,
            'variant': variant,
            'batch_size': record.get('concurrent_requests', 1),
            'throughput_tokens_per_sec': throughput,
            'hourly_cost': costs['hourly_cost'],
            'idle_multiplier': costs['idle_multiplier'],
        })
    except Exception:
        continue

df = pd.DataFrame(processed_data)
print(f"Processed {len(df)} records")
print(f"Hardware configs: {sorted(df['hardware_config'].unique())}")
print(f"Throughput range: {df['throughput_tokens_per_sec'].min():.1f} - {df['throughput_tokens_per_sec'].max():.1f} tokens/sec")

In [ ]:
df

In [ ]:
# Calculate cost per token metrics across different token workloads
def calculate_cost_per_token_metrics(df, tokens_per_hour):
    """Calculate cost per token for a given token workload per hour."""
    results = []

    for hw_config in df['hardware_config'].unique():
        config_data = df[df['hardware_config'] == hw_config]
        avg_throughput = config_data['throughput_tokens_per_sec'].mean()
        hourly_cost = config_data['hourly_cost'].iloc[0]
        idle_multiplier = config_data['idle_multiplier'].iloc[0]
        variant = config_data['variant'].iloc[0]

        # Calculate processing time needed
        processing_time_seconds = tokens_per_hour / avg_throughput
        processing_time_hours = processing_time_seconds / 3600

        # Calculate idle time (remaining time in the hour)
        idle_time_hours = max(0, 1.0 - processing_time_hours)

        # Calculate costs
        processing_cost_per_hour = processing_time_hours * hourly_cost
        idle_cost_per_hour = idle_time_hours * hourly_cost * idle_multiplier
        total_hourly_cost = processing_cost_per_hour + idle_cost_per_hour

        # Cost per token is total hourly cost divided by tokens processed per hour
        cost_per_token = total_hourly_cost / tokens_per_hour if tokens_per_hour > 0 else 0

        # Hardware utilization
        utilization_percent = min(100, (processing_time_hours / 1.0) * 100)

        results.append({
            'tokens_per_hour': tokens_per_hour,
            'hardware_config': hw_config,
            'variant': variant,
            'cost_per_token': cost_per_token,
            'total_hourly_cost': total_hourly_cost,
            'utilization_percent': utilization_percent,
            'processing_cost_per_hour': processing_cost_per_hour,
            'idle_cost_per_hour': idle_cost_per_hour,
            'avg_throughput': avg_throughput,
            'processing_time_hours': processing_time_hours,
        })

    return pd.DataFrame(results)

# Analyze multiple token workload levels
token_workload_levels = [100, 500, 1000, 5000, 10000, 50000, 100000, 500000]
all_token_data = []

for tph in token_workload_levels:
    token_result = calculate_cost_per_token_metrics(df, tph)
    all_token_data.append(token_result)

# Combine all token workload data
combined_token_df = pd.concat(all_token_data, ignore_index=True)

print(f"\nAnalyzed {len(token_workload_levels)} token workload levels")
print(f"Token workload range: {min(token_workload_levels):,} - {max(token_workload_levels):,} tokens/hour")
print(f"Total data points: {len(combined_token_df)}")

In [ ]:
# Create color palette
configs = sorted(df['hardware_config'].unique())
cpu_configs = [c for c in configs if c.startswith('CPU')]
gpu_configs = [c for c in configs if c.startswith('GPU')]

cpu_colors = plt.cm.Oranges(np.linspace(0.4, 0.9, len(cpu_configs)))
gpu_colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(gpu_configs)))

color_map = {}
for i, config in enumerate(cpu_configs):
    color_map[config] = cpu_colors[i]
for i, config in enumerate(gpu_configs):
    color_map[config] = gpu_colors[i]

# Visualization 1: Cost per Token vs Token Workload
plt.figure(figsize=(16, 10))

for hw_config in sorted(configs):
    config_data = combined_token_df[combined_token_df['hardware_config'] == hw_config]
    plt.plot(config_data['tokens_per_hour'], config_data['cost_per_token'],
            marker='o', linewidth=3, markersize=8,
            label=hw_config, color=color_map[hw_config])

plt.xlabel('Tokens per Hour', fontsize=14)
plt.ylabel('Cost per Token ($)', fontsize=14)
plt.title('Cost per Token vs Token Workload\nQ4_K_M Model - Optimal Hardware Selection',
          fontsize=16, fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.xscale('log')
plt.yscale('log')

# Add workload level annotations
for level in [1000, 10000, 100000]:
    plt.axvline(x=level, color='red', linestyle='--', alpha=0.5)
    plt.text(level, plt.ylim()[1]*0.8, f'{level:,}\ntokens/hr', ha='center', fontsize=10,
             bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

plt.tight_layout()
plt.show()

print("\n📊 Cost per Token Analysis: More accurate than cost per request!")
print("Shows true processing efficiency independent of request size.")

In [ ]:
# Find optimal hardware for each token workload level
print("\n" + "="*80)
print("OPTIMAL HARDWARE SELECTION BY TOKEN WORKLOAD")
print("="*80)

optimal_token_selections = []
for tph in token_workload_levels:
    workload_data = combined_token_df[combined_token_df['tokens_per_hour'] == tph]
    best_config = workload_data.loc[workload_data['cost_per_token'].idxmin()]

    optimal_token_selections.append({
        'tokens_per_hour': tph,
        'optimal_hardware': best_config['hardware_config'],
        'cost_per_token': best_config['cost_per_token'],
        'utilization': best_config['utilization_percent'],
        'variant': best_config['variant']
    })

    print(f"{tph:6,} tokens/hr: {best_config['hardware_config']:15} - "
          f"${best_config['cost_per_token']:.6f}/token ({best_config['utilization_percent']:5.1f}% util)")

optimal_token_df = pd.DataFrame(optimal_token_selections)

# Find CPU-GPU crossover point for tokens
cpu_token_workloads = optimal_token_df[optimal_token_df['variant'] == 'cpu']['tokens_per_hour'].tolist()
gpu_token_workloads = optimal_token_df[optimal_token_df['variant'] == 'cuda']['tokens_per_hour'].tolist()

if cpu_token_workloads and gpu_token_workloads:
    token_crossover_point = min(gpu_token_workloads)
    print(f"\n🎯 CPU-GPU CROSSOVER POINT: ~{token_crossover_point:,} tokens/hour")
    print(f"   Below {token_crossover_point:,} tokens/hr: Use CPU")
    print(f"   Above {token_crossover_point:,} tokens/hr: Use GPU")
else:
    print("\n🎯 No clear crossover point found in this range")

In [ ]:
# Visualization 2: Cost per Token Heatmap
plt.figure(figsize=(14, 10))

# Create pivot table for heatmap
heatmap_token_data = combined_token_df.pivot_table(
    values='cost_per_token',
    index='hardware_config',
    columns='tokens_per_hour',
    aggfunc='mean'
)

# Create heatmap with better formatting
sns.heatmap(heatmap_token_data, annot=True, fmt='.2e', cmap='RdYlGn_r',
            cbar_kws={'label': 'Cost per Token ($)'})

plt.title('Cost per Token Heatmap\nHardware Config vs Token Workload Level',
          fontsize=16, fontweight='bold')
plt.xlabel('Tokens per Hour', fontsize=14)
plt.ylabel('Hardware Configuration', fontsize=14)
plt.xticks(rotation=45)
plt.yticks(rotation=0)

# Format x-axis labels to show K/M notation
ax = plt.gca()
xlabels = []
for label in ax.get_xticklabels():
    val = int(float(label.get_text()))
    if val >= 1000000:
        xlabels.append(f'{val//1000000}M')
    elif val >= 1000:
        xlabels.append(f'{val//1000}K')
    else:
        xlabels.append(str(val))
ax.set_xticklabels(xlabels)

plt.tight_layout()
plt.show()

print("\n🔥 Heatmap shows cost per token patterns - Green = Lower cost, Red = Higher cost")

In [ ]:
# Detailed cost analysis for key workload levels
print("\n" + "="*90)
print("DETAILED COST PER TOKEN ANALYSIS")
print("="*90)

key_workloads = [1000, 10000, 100000]  # 1K, 10K, 100K tokens/hour

for tph in key_workloads:
    print(f"\n📊 {tph:,} TOKENS/HOUR ANALYSIS:")
    print("-" * 60)

    workload_data = combined_token_df[combined_token_df['tokens_per_hour'] == tph].sort_values('cost_per_token')

    print(f"{'Hardware':<15} {'$/Token':<12} {'Utilization':<12} {'$/Hour':<10}")
    print("-" * 60)

    for _, row in workload_data.iterrows():
        print(f"{row['hardware_config']:<15} "
              f"${row['cost_per_token']:<11.6f} "
              f"{row['utilization_percent']:<11.1f}% "
              f"${row['total_hourly_cost']:<9.3f}")

    # Show best CPU vs best GPU
    cpu_data = workload_data[workload_data['variant'] == 'cpu']
    gpu_data = workload_data[workload_data['variant'] == 'cuda']

    if len(cpu_data) > 0 and len(gpu_data) > 0:
        best_cpu = cpu_data.iloc[0]
        best_gpu = gpu_data.iloc[0]

        cpu_advantage = ((best_gpu['cost_per_token'] - best_cpu['cost_per_token']) / best_gpu['cost_per_token']) * 100

        print(f"\n  🏆 Best CPU: {best_cpu['hardware_config']} - ${best_cpu['cost_per_token']:.6f}/token")
        print(f"  🏆 Best GPU: {best_gpu['hardware_config']} - ${best_gpu['cost_per_token']:.6f}/token")

        if best_cpu['cost_per_token'] < best_gpu['cost_per_token']:
            print(f"  💰 CPU Advantage: {cpu_advantage:.1f}% cheaper per token")
        else:
            gpu_advantage = ((best_cpu['cost_per_token'] - best_gpu['cost_per_token']) / best_cpu['cost_per_token']) * 100
            print(f"  💰 GPU Advantage: {gpu_advantage:.1f}% cheaper per token")

In [ ]:
# Summary and system optimization recommendations
print("\n" + "="*80)
print("COST PER TOKEN OPTIMIZATION RECOMMENDATIONS")
print("="*80)

print(f"\n🎯 KEY FINDINGS:")
print(f"• Cost per token provides more accurate optimization than cost per request")
print(f"• Independent of request size variations")
print(f"• Better reflects true processing efficiency")

if cpu_token_workloads and gpu_token_workloads:
    print(f"\n📊 OPTIMIZATION THRESHOLDS (Cost per Token):")
    print(f"• Low workload (<{token_crossover_point:,} tokens/hr): Use CPU configurations")
    print(f"• High workload (>{token_crossover_point:,} tokens/hr): Use GPU configurations")
    print(f"• Monitor zone ({token_crossover_point//2:,}-{token_crossover_point*2:,} tokens/hr): Dynamic switching")

print(f"\n⚡ SYSTEM METRICS TO MONITOR:")
print(f"1. Tokens processed per hour (rolling average)")
print(f"2. Cost per token (real-time calculation)")
print(f"3. Hardware utilization percentage")
print(f"4. Processing queue depth")

print(f"\n🔧 IMPLEMENTATION STRATEGY:")
print(f"1. Track token throughput instead of request count")
print(f"2. Calculate cost per token in real-time")
print(f"3. Switch hardware when cost per token savings > threshold")
print(f"4. Use token-based workload prediction")
print(f"5. Optimize for token processing efficiency")

# Calculate potential savings
total_savings = 0
comparisons = 0

for tph in token_workload_levels:
    workload_data = combined_token_df[combined_token_df['tokens_per_hour'] == tph]
    if len(workload_data) > 1:
        best_cost = workload_data['cost_per_token'].min()
        worst_cost = workload_data['cost_per_token'].max()
        savings = ((worst_cost - best_cost) / worst_cost) * 100
        total_savings += savings
        comparisons += 1

avg_savings = total_savings / comparisons if comparisons > 0 else 0

print(f"\n💰 EXPECTED BENEFITS:")
print(f"• Average {avg_savings:.0f}% cost per token savings with optimal selection")
print(f"• More accurate cost predictions")
print(f"• Better resource utilization")
print(f"• Standardized optimization across different request patterns")

# Save results
combined_token_df.to_csv('cost_per_token_analysis_results.csv', index=False)
optimal_token_df.to_csv('optimal_hardware_by_tokens.csv', index=False)
print(f"\n💾 Results saved to CSV files for system implementation")
print(f"\n✅ Cost per token analysis provides superior optimization metrics!")